# PIEC GCD Testing Notebook

This notebook follows the AMR testing workflow for **Galvanostatic Charge-Discharge (GCD)** cycling. It uses the same PIEC `GCD` class and current-source protocol with physical or virtual SMUs. The instrument address defaults to `VIRTUAL` for the demonstration.

*COPY THIS NOTEBOOK AND RUN IN A LOCAL DIRECTORY TO PREVENT IT FROM BEING OVERWRITTEN BY GITHUB*

Install this checkout first with `python -m pip install -e .`, then select that Python environment as the notebook kernel. Run cells from top to bottom. Files go into `scratch/gcd_notebook`, relative to the notebook working directory.

### Imports

In [ ]:
from pathlib import Path

from piec.drivers.sourcemeter.keithley2400 import Keithley2400
from piec.drivers.virtual_instrument import VirtualInstrument
from piec.measurement.gcd import GCD
from piec.simulation.battery import Battery
from piec.analysis.utilities import standard_csv_to_metadata_and_data

import matplotlib.pyplot as plt

### Instrument Setup

GCD uses one **Source Measure Unit (SMU)** to source positive current during charge, source negative current during discharge, and read terminal voltage and current. Keep `smu_address = "VIRTUAL"` to run without hardware. To use a physical Keithley 2400, set its VISA address in the next cell (for example, `GPIB0::24::INSTR`). Select another compatible PIEC SMU driver if appropriate.

Only virtual instruments receive a `Battery` sample and accelerated time. Physical instruments use wall-clock sampling and the attached cell. The protocol requires an SMU that can source/sink the selected current and explicitly supports high-impedance (`HIMP`) output-off mode. GCD selects HIMP before disabling output and leaves it selected on cleanup; it does not reset the SMU while a battery may remain connected.

In [ ]:
smu_address = "VIRTUAL"  # Replace with your SMU VISA address for physical operation.
smu = Keithley2400(smu_address)
virtual = isinstance(smu, VirtualInstrument)
print("Sourcemeter:", smu.idn())
print("Timing:", "accelerated simulation" if virtual else "wall clock")

### GCD Measurement

Each cycle charges at `+current` until `voltage_upper`, then discharges at `-current` until `voltage_lower`. The first charge starts at the attached cell state (or `initial_soc` in simulation).

- **current**: positive current magnitude (A).
- **voltage_lower / voltage_upper**: terminal-voltage cutoffs (V).
- **n_cycles**: number of complete charge/discharge pairs.
- **sample_interval**: seconds between readings. Cutoffs can overshoot by one sampling interval.
- **max_half_cycle_time**: timeout for an unreachable cutoff.
- **voltage_compliance**: SMU limit (V), greater than both cutoff magnitudes.
- **capacity_ah / initial_soc**: virtual-only battery capacity (Ah) and starting charge fraction (0–1).

The demonstration battery uses coulomb counting, a linear open-circuit voltage curve, and series resistance. It does not model chemistry, aging, or temperature.

In [ ]:
### VARIABLE DEFINITIONS ###

path = Path("scratch/gcd_notebook")
current = 0.01                # A: +10 mA charge, -10 mA discharge
voltage_lower = 3.1           # V
voltage_upper = 4.1           # V
n_cycles = 3
sample_interval = 1.0         # seconds
max_half_cycle_time = 1000.0  # seconds
voltage_compliance = 5.0      # V
sense_mode = "2W"
capacity_ah = 0.001           # virtual-only: 1 mAh demo cell
initial_soc = 0.2
save_plots = True

In [ ]:
### RUN ###

original_sample = smu.sample if virtual else None
experiment = None
try:
    if virtual:
        # Only the virtual SMU receives a fresh simulated battery.
        smu.sample = Battery(capacity_ah=capacity_ah, initial_soc=initial_soc)
    experiment = GCD(
        sourcemeter=smu, current=current,
        voltage_lower=voltage_lower, voltage_upper=voltage_upper,
        n_cycles=n_cycles, sample_interval=sample_interval,
        max_half_cycle_time=max_half_cycle_time,
        voltage_compliance=voltage_compliance, sense_mode=sense_mode,
        virtual_time=virtual, save_dir=path,
        show_plots=False, save_plots=save_plots,
    )
    # The same protocol handles configuration, acquisition, HIMP cleanup and saving.
    experiment.run_experiment()
    print(f"Completed {experiment.completed_cycles} cycles")
    print("Data:", experiment.filename)
    print("Plot:", experiment.plot_filename)
finally:
    if virtual:
        VirtualInstrument.set_virtual_sample(original_sample)

### Saved Data and Metadata

PIEC’s standard CSV contains measurement parameters and metadata followed by raw readings: `time (s)`, `voltage (V)`, `current (A)`, `cycle`, and `phase`. Reload the saved file for analysis, as in the AMR notebook. Failed runs retain partial data with `status="aborted"`.

In [ ]:
gcd_metadata, gcd_df = standard_csv_to_metadata_and_data(experiment.filename)
gcd_df.head()

In [ ]:
gcd_metadata[["mtype", "status", "completed_cycles", "current", "voltage_lower", "voltage_upper"]]

### Plot Voltage versus Time

In [ ]:
ax = gcd_df.plot(x="time (s)", y="voltage (V)", figsize=(10, 5),
                 title="GCD Cycling", legend=False)
ax.axhline(voltage_lower, color="gray", linestyle="--", label="Voltage cutoffs")
ax.axhline(voltage_upper, color="gray", linestyle="--")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (V)")
ax.legend()
plt.tight_layout()
plt.show()

### Cycle Summary

In [ ]:
gcd_df.groupby(["cycle", "phase"], sort=False).agg(
    start_time_s=("time (s)", "min"),
    end_time_s=("time (s)", "max"),
    minimum_voltage_V=("voltage (V)", "min"),
    maximum_voltage_V=("voltage (V)", "max"),
    mean_current_A=("current (A)", "mean"),
)

### Cleanup

`run_experiment()` disables output, clears the current setpoint, and leaves HIMP selected on completion, Stop, timeout, or an acquisition error. `safe_close()` repeats that shutdown without resetting to the instrument default off mode. The run cell restores the shared sample only for virtual instruments.

To repeat the measurement, rerun the run cell or call `experiment.run_experiment()` again. A reused experiment clears stale Stop/Pause flags, starts elapsed time at zero, and creates a new CSV. In simulation, rerun the run cell to attach a fresh battery; a direct rerun needs the battery attached again.

In [ ]:
if experiment is not None:
    experiment.safe_close()
print("Sourcemeter output disabled with HIMP retained; no reset issued.")